# Data Wrangling & Exploratory Data Analysis
**UMKM Bersama -- CC26-PSU328 | Data Science Learning Path**

Notebook ini mencakup:
1. Load & inspeksi awal dataset
2. Assessing data (identifikasi masalah)
3. Cleaning data (penanganan masalah)
4. Exploratory Data Analysis (EDA)
5. Menjawab business questions

Output notebook ini adalah dataset bersih yang siap digunakan untuk tahap pemodelan:
Cash Flow Forecasting, Anomaly Detection, dan BCG Matrix Clustering.

---
## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.0f}'.format)

# ============================================================
# PALET WARNA GLOBAL (konsisten di seluruh notebook)
# ============================================================
# Palet utama: kombinasi biru-teal-oranye-merah yang serasi
PALET = {
    'primary':   '#2D6A8E',   # biru tua
    'secondary': '#3FA39B',   # teal
    'accent':    '#E8A33D',   # oranye/amber
    'danger':    '#D96459',   # merah bata
    'neutral':   '#8E9AAF',   # abu kebiruan
    'success':   '#5B9279',   # hijau sage
}

# Urutan warna untuk kategori (dipakai berulang agar konsisten)
PALET_LIST = ['#2D6A8E', '#3FA39B', '#E8A33D', '#D96459', '#5B9279', '#8E9AAF', '#9B6A9E', '#C97B84']

# Palet khusus untuk pemasukan vs pengeluaran (dipakai konsisten)
WARNA_PEMASUKAN   = '#2D6A8E'   # biru
WARNA_PENGELUARAN = '#D96459'   # merah bata

# Colormap sekuensial untuk heatmap (selaras dengan palet)
CMAP_SEQ  = 'YlGnBu'
CMAP_CORR = 'RdBu_r'

# Set tema global
sns.set_theme(style='whitegrid')
sns.set_palette(PALET_LIST)
plt.rcParams['figure.dpi']      = 120
plt.rcParams['font.family']     = 'sans-serif'
plt.rcParams['axes.titlesize']  = 12
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.edgecolor']  = '#CCCCCC'
plt.rcParams['figure.facecolor'] = 'white'

OUTPUT_DIR = './dataset_bersih'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Library berhasil diimport.')
print('Palet warna global siap digunakan.')

---
## 2. Load Dataset

In [ ]:
# Sesuaikan path jika dataset ada di direktori lain
df_warung    = pd.read_csv('./dataset_sintetis/warung.csv')
df_produk    = pd.read_csv('./dataset_sintetis/produk.csv')
df_transaksi = pd.read_csv('./dataset_sintetis/transaksi.csv')

print(f'warung.csv    : {len(df_warung):,} baris, {df_warung.shape[1]} kolom')
print(f'produk.csv    : {len(df_produk):,} baris, {df_produk.shape[1]} kolom')
print(f'transaksi.csv : {len(df_transaksi):,} baris, {df_transaksi.shape[1]} kolom')

In [ ]:
df_transaksi.head(10)

In [ ]:
df_transaksi.info()

In [ ]:
df_transaksi.describe(include='all')

---
## 3. Assessing Data
Identifikasi seluruh masalah data sebelum melakukan penanganan.

### 3.1 Missing Values

In [ ]:
def cek_missing(df, nama):
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(2)
    hasil   = pd.DataFrame({'jumlah_missing': missing, 'persentase (%)': pct})
    hasil   = hasil[hasil['jumlah_missing'] > 0].sort_values('jumlah_missing', ascending=False)
    print(f'--- Missing Values: {nama} ---')
    print(hasil if len(hasil) > 0 else 'Tidak ada missing values.')
    print()

cek_missing(df_warung,    'warung')
cek_missing(df_produk,    'produk')
cek_missing(df_transaksi, 'transaksi')

### 3.2 Duplikasi

In [ ]:
dup_total = df_transaksi.duplicated().sum()
dup_id    = df_transaksi.duplicated(subset=['id_transaksi']).sum()

print(f'Baris duplikat penuh         : {dup_total:,}')
print(f'Duplikat pada id_transaksi   : {dup_id:,}')

if dup_total > 0:
    print('\nContoh baris duplikat:')
    display(df_transaksi[df_transaksi.duplicated(keep=False)].head(6))

### 3.3 Inkonsistensi Format

In [ ]:
print('Nilai unik kolom jenis:')
print(df_transaksi['jenis'].value_counts())
print()
print('Nilai unik kolom metode_bayar:')
print(df_transaksi['metode_bayar'].value_counts())
print()
print('Nilai unik kolom kategori:')
print(df_transaksi['kategori'].value_counts())

In [ ]:
# Deteksi inkonsistensi format tanggal
def cek_format_tanggal(series):
    formats = {'%Y-%m-%d': 0, '%d/%m/%Y': 0, '%d-%m-%Y': 0, '%Y/%m/%d': 0, 'lainnya': 0}
    for val in series.dropna():
        matched = False
        for fmt in list(formats.keys())[:-1]:
            try:
                pd.to_datetime(val, format=fmt)
                formats[fmt] += 1
                matched = True
                break
            except:
                continue
        if not matched:
            formats['lainnya'] += 1
    return formats

format_tanggal = cek_format_tanggal(df_transaksi['tanggal'])
print('Distribusi format tanggal:')
for fmt, count in format_tanggal.items():
    if count > 0:
        print(f'  {fmt}: {count:,} baris')

### 3.4 Outlier Nominal

In [ ]:
# Cek nominal yang tidak wajar
print('Statistik kolom nominal:')
print(df_transaksi['nominal'].describe())
print()

# Nominal nol atau negatif
n_non_positif = (df_transaksi['nominal'] <= 0).sum()
print(f'Baris dengan nominal <= 0 : {n_non_positif}')

# Outlier dengan IQR
Q1  = df_transaksi['nominal'].quantile(0.25)
Q3  = df_transaksi['nominal'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR
n_outlier = ((df_transaksi['nominal'] < lower_bound) | (df_transaksi['nominal'] > upper_bound)).sum()
print(f'Outlier (IQR 3x)          : {n_outlier}')
print(f'Batas bawah               : {lower_bound:,.0f}')
print(f'Batas atas                : {upper_bound:,.0f}')

print('\nContoh baris dengan nominal sangat besar:')
display(df_transaksi[df_transaksi['nominal'] > upper_bound][['id_transaksi', 'tanggal', 'jenis', 'kategori', 'nominal']].head(10))

### 3.5 Referential Integrity

In [ ]:
# Cek id_produk yang tidak ada di tabel produk
valid_produk = set(df_produk['id_produk'].unique())
transaksi_penjualan = df_transaksi[df_transaksi['id_produk'].notna()]
invalid_produk = transaksi_penjualan[~transaksi_penjualan['id_produk'].isin(valid_produk)]
print(f'Baris dengan id_produk tidak valid: {len(invalid_produk)}')
print(invalid_produk['id_produk'].value_counts())

### 3.6 Inkonsistensi Logika Bisnis

In [ ]:
# Pemasukan seharusnya hanya berkategori Penjualan
# Pengeluaran seharusnya hanya berkategori HPP/Operasional/Overhead
jenis_bersih_map = {
    'pemasukan': 'Pemasukan', 'PEMASUKAN': 'Pemasukan', 'Pemasokan': 'Pemasukan',
    'income': 'Pemasukan', 'masuk': 'Pemasukan',
    'pengeluaran': 'Pengeluaran', 'PENGELUARAN': 'Pengeluaran', 'Pengeluaaran': 'Pengeluaran',
    'expense': 'Pengeluaran', 'keluar': 'Pengeluaran',
}
df_transaksi['jenis_bersih'] = df_transaksi['jenis'].replace(jenis_bersih_map).fillna(df_transaksi['jenis'])

inkonsisten = df_transaksi[
    ((df_transaksi['jenis_bersih'] == 'Pemasukan') & (df_transaksi['kategori'] != 'Penjualan')) |
    ((df_transaksi['jenis_bersih'] == 'Pengeluaran') & (df_transaksi['kategori'] == 'Penjualan'))
]
print(f'Baris dengan inkonsistensi jenis vs kategori: {len(inkonsisten)}')
display(inkonsisten[['id_transaksi', 'jenis', 'kategori']].head(10))

---
## 4. Cleaning Data
Penanganan seluruh masalah yang ditemukan pada tahap assessing.

### 4.1 Hapus Duplikasi

In [ ]:
before = len(df_transaksi)
df_transaksi = df_transaksi.drop_duplicates()
df_transaksi = df_transaksi.drop_duplicates(subset=['id_transaksi'])
print(f'Baris dihapus (duplikat): {before - len(df_transaksi):,}')
print(f'Sisa baris              : {len(df_transaksi):,}')

### 4.2 Standardisasi Format Tanggal

In [ ]:
df_transaksi['tanggal'] = pd.to_datetime(df_transaksi['tanggal'], dayfirst=True, errors='coerce')

n_invalid_tanggal = df_transaksi['tanggal'].isnull().sum()
print(f'Tanggal tidak bisa diparsing: {n_invalid_tanggal}')

# Drop baris dengan tanggal tidak valid (tidak bisa diselamatkan)
df_transaksi = df_transaksi.dropna(subset=['tanggal'])
print(f'Sisa baris setelah drop tanggal invalid: {len(df_transaksi):,}')

### 4.3 Standardisasi Kolom Jenis

In [ ]:
jenis_map = {
    'pemasukan': 'Pemasukan', 'PEMASUKAN': 'Pemasukan', 'Pemasokan': 'Pemasukan',
    'income': 'Pemasukan', 'masuk': 'Pemasukan',
    'pengeluaran': 'Pengeluaran', 'PENGELUARAN': 'Pengeluaran', 'Pengeluaaran': 'Pengeluaran',
    'expense': 'Pengeluaran', 'keluar': 'Pengeluaran',
    'Pemasukan': 'Pemasukan', 'Pengeluaran': 'Pengeluaran',
}

df_transaksi['jenis'] = df_transaksi['jenis'].map(jenis_map)

# Drop baris yang jenis-nya tidak bisa dipetakan
before = len(df_transaksi)
df_transaksi = df_transaksi.dropna(subset=['jenis'])
print(f'Baris dengan jenis tidak valid yang di-drop: {before - len(df_transaksi)}')
print('Distribusi jenis setelah cleaning:')
print(df_transaksi['jenis'].value_counts())

### 4.4 Standardisasi Kolom Metode Bayar

In [ ]:
metode_map = {
    'Cash': 'Cash', 'cash': 'Cash', 'CASH': 'Cash', 'tunai': 'Cash', 'Tunai': 'Cash', 'cash ': 'Cash',
    'Transfer': 'Transfer', 'transfer': 'Transfer', 'TRANSFER': 'Transfer', 'tf': 'Transfer', 'TF': 'Transfer', 'Trasfer': 'Transfer',
    'QRIS': 'QRIS', 'qris': 'QRIS', 'Qris': 'QRIS', 'scan': 'QRIS', 'qr': 'QRIS',
}

df_transaksi['metode_bayar'] = df_transaksi['metode_bayar'].map(metode_map)

# Impute missing metode_bayar dengan modus per jenis transaksi
modus_metode = df_transaksi.groupby('jenis')['metode_bayar'].agg(lambda x: x.mode()[0] if not x.mode().empty else 'Cash')
for jenis_val, modus_val in modus_metode.items():
    mask = (df_transaksi['jenis'] == jenis_val) & (df_transaksi['metode_bayar'].isnull())
    df_transaksi.loc[mask, 'metode_bayar'] = modus_val

print('Distribusi metode_bayar setelah cleaning:')
print(df_transaksi['metode_bayar'].value_counts())

### 4.5 Penanganan Nominal Bermasalah

In [ ]:
# Drop baris dengan nominal <= 0
before = len(df_transaksi)
df_transaksi = df_transaksi[df_transaksi['nominal'] > 0]
print(f'Baris dengan nominal <= 0 yang di-drop: {before - len(df_transaksi)}')

# Cast nominal ke float agar operasi median dan clip tidak error dtype
df_transaksi['nominal'] = df_transaksi['nominal'].astype(float)

# Impute missing nominal dengan median per kategori
median_per_kat = df_transaksi.groupby('kategori')['nominal'].median()
for kat, median_val in median_per_kat.items():
    mask = (df_transaksi['kategori'] == kat) & (df_transaksi['nominal'].isnull())
    df_transaksi.loc[mask, 'nominal'] = median_val

# Cap outlier ekstrem dengan batas IQR 3x per kategori
# Pakai transform agar kolom grouping (kategori) tidak hilang
batas_atas = df_transaksi.groupby('kategori')['nominal'].transform(
    lambda x: x.quantile(0.75) + 3 * (x.quantile(0.75) - x.quantile(0.25))
)
df_transaksi['nominal'] = np.minimum(df_transaksi['nominal'], batas_atas)

print(f'Sisa baris setelah penanganan nominal: {len(df_transaksi):,}')
print('\nStatistik nominal setelah cleaning:')
print(df_transaksi['nominal'].describe())

### 4.6 Penanganan Missing jam_transaksi

In [ ]:
# jam_transaksi dibutuhkan untuk anomaly detection dan heatmap
# Baris dengan jam kosong di-flag, tidak di-drop agar forecasting tidak kehilangan data
df_transaksi['jam_missing'] = df_transaksi['jam_transaksi'].isnull().astype(int)

n_jam_missing = df_transaksi['jam_missing'].sum()
print(f'Baris dengan jam_transaksi kosong: {n_jam_missing:,}')
print('Baris ini tetap dipertahankan untuk forecasting, namun akan dieksklusi saat training Anomaly Detection dan heatmap.')

### 4.7 Perbaiki Referential Integrity & Inkonsistensi Logika

In [ ]:
# Drop baris dengan id_produk tidak valid
valid_produk = set(df_produk['id_produk'].unique())
mask_invalid = df_transaksi['id_produk'].notna() & ~df_transaksi['id_produk'].isin(valid_produk)
before = len(df_transaksi)
df_transaksi = df_transaksi[~mask_invalid]
print(f'Baris dengan id_produk tidak valid yang di-drop: {before - len(df_transaksi)}')

# Perbaiki inkonsistensi kategori vs jenis
# Pemasukan harus kategori Penjualan
mask_fix = (df_transaksi['jenis'] == 'Pemasukan') & (df_transaksi['kategori'] != 'Penjualan')
df_transaksi.loc[mask_fix, 'kategori'] = 'Penjualan'
print(f'Baris kategori diperbaiki (pemasukan -> Penjualan): {mask_fix.sum()}')

### 4.8 Tambah Derived Columns

In [ ]:
df_transaksi['hari_dalam_minggu'] = df_transaksi['tanggal'].dt.dayofweek  # 0=Senin, 6=Minggu
df_transaksi['nama_hari']         = df_transaksi['tanggal'].dt.day_name()
df_transaksi['bulan']             = df_transaksi['tanggal'].dt.to_period('M').astype(str)
df_transaksi['minggu_ke']         = df_transaksi['tanggal'].dt.to_period('W').astype(str)
df_transaksi['is_awal_bulan']     = (df_transaksi['tanggal'].dt.day <= 7).astype(int)
df_transaksi['is_weekend']        = (df_transaksi['hari_dalam_minggu'] >= 5).astype(int)

# Jam encode (hanya untuk baris yang ada jam_transaksinya)
df_transaksi['jam_transaksi'] = pd.to_datetime(df_transaksi['jam_transaksi'], format='%H:%M:%S', errors='coerce')
df_transaksi['jam_encoded']   = df_transaksi['jam_transaksi'].dt.hour

print('Derived columns berhasil ditambahkan.')
print(df_transaksi[['tanggal', 'hari_dalam_minggu', 'nama_hari', 'bulan', 'is_awal_bulan', 'is_weekend', 'jam_encoded']].head())

### 4.9 Ringkasan Hasil Cleaning

In [ ]:
print('==== RINGKASAN HASIL CLEANING ====')
print(f'Total baris final    : {len(df_transaksi):,}')
print(f'Kolom                : {list(df_transaksi.columns)}')
print()
print('Missing values setelah cleaning:')
missing_after = df_transaksi.isnull().sum()
print(missing_after[missing_after > 0])

### 4.10 Export Dataset Bersih

In [ ]:
df_transaksi.to_csv(f'{OUTPUT_DIR}/transaksi_bersih.csv', index=False, encoding='utf-8')
df_produk.to_csv(f'{OUTPUT_DIR}/produk_bersih.csv',       index=False, encoding='utf-8')
df_warung.to_csv(f'{OUTPUT_DIR}/warung_bersih.csv',       index=False, encoding='utf-8')
print('Dataset bersih berhasil disimpan di:', OUTPUT_DIR)

---
## 5. Exploratory Data Analysis
EDA dilakukan untuk memahami pola, distribusi, dan karakteristik data sebelum masuk ke pemodelan.

### 5.1 Distribusi Nominal per Kategori

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

kategori_list = df_transaksi['kategori'].unique()
for i, kat in enumerate(kategori_list[:4]):
    data = df_transaksi[df_transaksi['kategori'] == kat]['nominal']
    axes[i].hist(data, bins=40, color=PALET_LIST[i], edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribusi Nominal -- {kat}', fontsize=12)
    axes[i].set_xlabel('Nominal (Rp)')
    axes[i].set_ylabel('Frekuensi')
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp{x/1000:.0f}k'))

plt.suptitle('Distribusi Nominal Transaksi per Kategori', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 5.2 Tren Pemasukan dan Pengeluaran Harian

In [ ]:
harian = df_transaksi.groupby(['tanggal', 'jenis'])['nominal'].sum().unstack(fill_value=0).reset_index()

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(harian['tanggal'], harian.get('Pemasukan', 0),   label='Pemasukan',   color=WARNA_PEMASUKAN, linewidth=1.3, alpha=0.9)
ax.plot(harian['tanggal'], harian.get('Pengeluaran', 0), label='Pengeluaran', color=WARNA_PENGELUARAN, linewidth=1.3, alpha=0.9)
ax.set_title('Tren Pemasukan dan Pengeluaran Harian (semua warung)', fontsize=13)
ax.set_xlabel('Tanggal')
ax.set_ylabel('Total Nominal (Rp)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp{x/1_000_000:.1f}M'))
ax.legend(frameon=True, loc='upper right')
plt.tight_layout()
plt.show()

### 5.3 Tren Pemasukan Bulanan per Warung

In [ ]:
bulanan = df_transaksi[
    df_transaksi['jenis'] == 'Pemasukan'
].groupby(['bulan', 'id_warung'])['nominal'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
for i, warung_id in enumerate(bulanan['id_warung'].unique()):
    data = bulanan[bulanan['id_warung'] == warung_id]
    nama = df_warung[df_warung['id_warung'] == warung_id]['nama_warung'].values[0]
    ax.plot(data['bulan'], data['nominal'], marker='o', label=nama,
            color=PALET_LIST[i % len(PALET_LIST)], linewidth=1.8, markersize=5)

ax.set_title('Tren Pemasukan Bulanan per Warung', fontsize=13)
ax.set_xlabel('Bulan')
ax.set_ylabel('Total Pemasukan (Rp)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp{x/1_000_000:.1f}M'))
# Legend dipindah ke luar plot agar tidak menutupi garis
ax.legend(fontsize=9, loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 5.4 Pola Penjualan per Hari dalam Seminggu

In [ ]:
urutan_hari = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
label_hari  = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']

per_hari = df_transaksi[
    df_transaksi['jenis'] == 'Pemasukan'
].groupby('nama_hari')['nominal'].mean().reindex(urutan_hari)

fig, ax = plt.subplots(figsize=(10, 5))
# Gradasi satu warna (biru) agar serasi, weekend di-highlight dengan teal
warna_hari = [PALET['primary']] * 5 + [PALET['secondary']] * 2
bars = ax.bar(label_hari, per_hari.values, color=warna_hari, alpha=0.9)
ax.set_title('Rata-rata Pemasukan per Hari dalam Seminggu', fontsize=13)
ax.set_ylabel('Rata-rata Pemasukan (Rp)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp{x/1000:.0f}k'))
ax.margins(y=0.12)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + (per_hari.max()*0.01),
            f'Rp{bar.get_height()/1000:.0f}k', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

### 5.5 Heatmap Penjualan per Jam x Hari

In [ ]:
df_heatmap = df_transaksi[
    (df_transaksi['jenis'] == 'Pemasukan') &
    (df_transaksi['jam_encoded'].notna())
].copy()

df_heatmap['jam_encoded'] = df_heatmap['jam_encoded'].astype(int)

pivot = df_heatmap.groupby(['hari_dalam_minggu', 'jam_encoded'])['nominal'].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    pivot,
    ax=ax,
    cmap=CMAP_SEQ,
    linewidths=0.3,
    linecolor='white',
    yticklabels=label_hari,
    cbar_kws={'label': 'Total Penjualan (Rp)'}
)
ax.set_title('Heatmap Total Penjualan per Jam x Hari', fontsize=13)
ax.set_xlabel('Jam')
ax.set_ylabel('Hari')
plt.tight_layout()
plt.show()

### 5.6 Komposisi Pengeluaran per Kategori

In [ ]:
pengeluaran = df_transaksi[
    df_transaksi['jenis'] == 'Pengeluaran'
].groupby('kategori')['nominal'].sum().sort_values(ascending=False)

# Warna konsisten per kategori pengeluaran
warna_pengeluaran = [PALET_LIST[i] for i in range(len(pengeluaran))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Bar chart horizontal ---
axes[0].barh(pengeluaran.index, pengeluaran.values, color=warna_pengeluaran, alpha=0.9)
axes[0].set_title('Total Pengeluaran per Kategori', fontsize=12)
axes[0].set_xlabel('Total Pengeluaran (Rp)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp{x/1_000_000:.0f}M'))
axes[0].invert_yaxis()

# --- Donut chart (pengganti pie, label tidak tumpang tindih) ---
# Pakai legend terpisah + autopct di dalam, bukan label di tepi
wedges, texts, autotexts = axes[1].pie(
    pengeluaran.values,
    autopct=lambda p: f'{p:.1f}%' if p >= 5 else '',
    colors=warna_pengeluaran,
    startangle=90,
    pctdistance=0.8,
    wedgeprops=dict(width=0.42, edgecolor='white', linewidth=2)
)
# Atur teks persentase agar terbaca
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')

# Legend terpisah di samping, bukan label menempel di pie
axes[1].legend(
    wedges,
    [f'{k} ({v/pengeluaran.sum()*100:.1f}%)' for k, v in pengeluaran.items()],
    title='Kategori',
    loc='center left',
    bbox_to_anchor=(0.95, 0.5),
    fontsize=9,
    frameon=True
)
axes[1].set_title('Proporsi Pengeluaran per Kategori', fontsize=12)

plt.suptitle('Komposisi Pengeluaran Warung Sembako', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.7 Distribusi Metode Pembayaran

In [ ]:
metode_dist = df_transaksi[
    df_transaksi['jenis'] == 'Pemasukan'
].groupby('metode_bayar')['nominal'].agg(['count', 'sum']).reset_index()
metode_dist.columns = ['metode_bayar', 'jumlah_transaksi', 'total_nominal']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Warna konsisten per metode bayar (dipetakan tetap)
warna_metode = {'Cash': PALET['primary'], 'Transfer': PALET['secondary'], 'QRIS': PALET['accent']}
colors_metode = [warna_metode.get(m, PALET['neutral']) for m in metode_dist['metode_bayar']]

axes[0].bar(metode_dist['metode_bayar'], metode_dist['jumlah_transaksi'], color=colors_metode, alpha=0.9)
axes[0].set_title('Jumlah Transaksi per Metode Bayar', fontsize=12)
axes[0].set_ylabel('Jumlah Transaksi')

axes[1].bar(metode_dist['metode_bayar'], metode_dist['total_nominal'], color=colors_metode, alpha=0.9)
axes[1].set_title('Total Nominal per Metode Bayar', fontsize=12)
axes[1].set_ylabel('Total Nominal (Rp)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp{x/1_000_000:.0f}M'))

plt.suptitle('Distribusi Metode Pembayaran', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.8 Top 10 Produk Terlaris (Volume)

In [ ]:
# Join transaksi dengan produk untuk dapatkan nama produk
df_join = df_transaksi[
    df_transaksi['jenis'] == 'Pemasukan'
].merge(df_produk[['id_produk', 'nama_produk', 'harga_jual', 'harga_pokok', 'kategori_produk']], 
        on='id_produk', how='left')

top_volume = df_join.groupby('nama_produk')['qty'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(11, 5.5))
# Gradasi biru-teal agar serasi
colors_grad = sns.color_palette('crest', 10)
bars = ax.barh(top_volume.index[::-1], top_volume.values[::-1], color=colors_grad, alpha=0.95)
ax.set_title('Top 10 Produk Terlaris Berdasarkan Volume Penjualan', fontsize=13)
ax.set_xlabel('Total Qty Terjual')
ax.margins(x=0.08)
for bar in bars:
    ax.text(bar.get_width() + (top_volume.max()*0.01), bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=8.5)
plt.tight_layout()
plt.show()

### 5.9 Perbandingan Margin per Produk

In [ ]:
# Kalkulasi margin per produk
df_margin = df_produk.copy()
df_margin['margin_pct'] = ((df_margin['harga_jual'] - df_margin['harga_pokok']) / df_margin['harga_jual'] * 100).round(1)

margin_per_kategori = df_margin.groupby('kategori_produk')['margin_pct'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
# Warna berdasarkan tingkat margin, tetap dalam palet (teal=tinggi, amber=sedang, merah=rendah)
colors = [PALET['secondary'] if m >= 15 else PALET['accent'] if m >= 10 else PALET['danger'] for m in margin_per_kategori.values]
bars = ax.bar(margin_per_kategori.index, margin_per_kategori.values, color=colors, alpha=0.9)
ax.set_title('Rata-rata Margin per Kategori Produk', fontsize=13)
ax.set_ylabel('Margin (%)')
ax.margins(y=0.12)
ax.axhline(y=margin_per_kategori.mean(), color=PALET['neutral'], linestyle='--', linewidth=1.2,
           label=f'Rata-rata: {margin_per_kategori.mean():.1f}%')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
ax.legend(frameon=True)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 5.10 Korelasi antar Variabel Numerik

In [ ]:
cols_corr = ['nominal', 'qty', 'hari_dalam_minggu', 'jam_encoded', 'is_awal_bulan', 'is_weekend']
df_corr   = df_transaksi[cols_corr].dropna()

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(
    df_corr.corr(),
    annot=True,
    fmt='.2f',
    cmap=CMAP_CORR,
    center=0,
    vmin=-1, vmax=1,
    ax=ax,
    linewidths=0.5,
    linecolor='white',
    square=True,
    cbar_kws={'shrink': 0.8, 'label': 'Koefisien Korelasi'}
)
ax.set_title('Heatmap Korelasi Variabel Numerik', fontsize=13)
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 6. Business Questions
Menjawab lima business questions yang telah didefinisikan di awal proyek.

### BQ 1: Berapa rata-rata laba bersih warung per minggu dan per bulan, dan bagaimana trennya dari waktu ke waktu?

In [ ]:
# Hitung laba bersih = pemasukan - pengeluaran per periode per warung
cashflow = df_transaksi.groupby(['bulan', 'id_warung', 'jenis'])['nominal'].sum().unstack(fill_value=0).reset_index()
cashflow.columns.name = None

if 'Pemasukan' not in cashflow.columns:
    cashflow['Pemasukan'] = 0
if 'Pengeluaran' not in cashflow.columns:
    cashflow['Pengeluaran'] = 0

cashflow['laba_bersih'] = cashflow['Pemasukan'] - cashflow['Pengeluaran']

rata_bulanan = cashflow.groupby('bulan')['laba_bersih'].mean().reset_index()

print('Rata-rata laba bersih per bulan (semua warung):')
for _, row in rata_bulanan.iterrows():
    print(f"  {row['bulan']}: Rp{row['laba_bersih']:,.0f}")

fig, ax = plt.subplots(figsize=(12, 4.5))
# Hijau sage untuk laba positif, merah bata untuk negatif (selaras palet)
bars = ax.bar(rata_bulanan['bulan'], rata_bulanan['laba_bersih'],
       color=[PALET['success'] if v >= 0 else PALET['danger'] for v in rata_bulanan['laba_bersih']], alpha=0.9)
ax.set_title('BQ1: Rata-rata Laba Bersih Bulanan per Warung', fontsize=13)
ax.set_ylabel('Laba Bersih (Rp)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp{x/1_000_000:.1f}M'))
ax.axhline(y=0, color='#555555', linewidth=0.8)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### BQ 2: Produk mana yang memiliki margin tertinggi dan volume tertinggi, apakah keduanya selalu sejalan?

In [ ]:
# Hitung margin dan total qty per produk
df_margin2 = df_produk[['id_produk', 'nama_produk', 'harga_jual', 'harga_pokok', 'kategori_produk']].copy()
df_margin2['margin_pct'] = ((df_margin2['harga_jual'] - df_margin2['harga_pokok']) / df_margin2['harga_jual'] * 100).round(1)

qty_per_produk = df_join.groupby('id_produk')['qty'].sum().reset_index()
qty_per_produk.columns = ['id_produk', 'qty_terjual']

df_bcg_prep = df_margin2.merge(qty_per_produk, on='id_produk', how='left').fillna(0)
df_bcg_prep = df_bcg_prep[df_bcg_prep['qty_terjual'] > 0]

# Scatter plot margin vs volume
fig, ax = plt.subplots(figsize=(13, 7))
kategori_unik = sorted(df_bcg_prep['kategori_produk'].unique())
palette = dict(zip(kategori_unik, PALET_LIST[:len(kategori_unik)]))

for kat in kategori_unik:
    subset = df_bcg_prep[df_bcg_prep['kategori_produk'] == kat]
    ax.scatter(subset['qty_terjual'], subset['margin_pct'], label=kat,
               color=palette[kat], s=70, alpha=0.85, edgecolors='white', linewidths=0.6)

# Garis median sebagai pembatas kuadran
median_qty    = df_bcg_prep['qty_terjual'].median()
median_margin = df_bcg_prep['margin_pct'].median()
ax.axvline(x=median_qty,    color=PALET['neutral'], linestyle='--', linewidth=1)
ax.axhline(y=median_margin, color=PALET['neutral'], linestyle='--', linewidth=1)

# Label produk
for _, row in df_bcg_prep.iterrows():
    ax.annotate(row['nama_produk'], (row['qty_terjual'], row['margin_pct']),
                fontsize=6.5, alpha=0.7, xytext=(4, 3), textcoords='offset points')

ax.set_title('BQ2: Margin vs Volume Penjualan per Produk (BCG Matrix Preview)', fontsize=13)
ax.set_xlabel('Total Qty Terjual')
ax.set_ylabel('Margin (%)')
# Legend di luar plot agar tidak menutupi titik
ax.legend(fontsize=9, loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=True, title='Kategori')
plt.tight_layout()
plt.show()

print('\nTop 5 produk berdasarkan margin:')
print(df_bcg_prep.nlargest(5, 'margin_pct')[['nama_produk', 'margin_pct', 'qty_terjual']].to_string(index=False))
print('\nTop 5 produk berdasarkan volume:')
print(df_bcg_prep.nlargest(5, 'qty_terjual')[['nama_produk', 'margin_pct', 'qty_terjual']].to_string(index=False))

### BQ 3: Pada jam dan hari apa transaksi penjualan paling tinggi terjadi?

In [ ]:
df_jam = df_transaksi[
    (df_transaksi['jenis'] == 'Pemasukan') &
    (df_transaksi['jam_encoded'].notna())
].copy()

# Jam paling ramai
per_jam = df_jam.groupby('jam_encoded')['nominal'].sum().sort_index()
jam_puncak = per_jam.idxmax()

# Hari paling ramai
per_hari2 = df_jam.groupby('nama_hari')['nominal'].sum().reindex(urutan_hari)
hari_puncak = per_hari2.idxmax()

print(f'Jam dengan penjualan tertinggi : {jam_puncak}:00 - {jam_puncak+1}:00')
print(f'Hari dengan penjualan tertinggi: {hari_puncak}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar per jam: highlight jam puncak dengan warna accent, sisanya primary
warna_jam = [PALET['accent'] if j == jam_puncak else PALET['primary'] for j in per_jam.index]
axes[0].bar(per_jam.index, per_jam.values, color=warna_jam, alpha=0.9)
axes[0].set_title('BQ3: Total Penjualan per Jam', fontsize=12)
axes[0].set_xlabel('Jam')
axes[0].set_ylabel('Total Penjualan (Rp)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp{x/1_000_000:.0f}M'))

# Bar per hari: highlight hari puncak
label_puncak = label_hari[urutan_hari.index(hari_puncak)]
warna_hari2 = [PALET['accent'] if h == label_puncak else PALET['secondary'] for h in label_hari]
axes[1].bar(label_hari, per_hari2.values, color=warna_hari2, alpha=0.9)
axes[1].set_title('BQ3: Total Penjualan per Hari', fontsize=12)
axes[1].set_xlabel('Hari')
axes[1].set_ylabel('Total Penjualan (Rp)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp{x/1_000_000:.0f}M'))

plt.suptitle('BQ3: Pola Jam dan Hari Penjualan Tertinggi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### BQ 4: Kategori pengeluaran mana yang paling dominan, dan apakah ada pengeluaran yang terindikasi tidak wajar?

In [ ]:
df_keluar = df_transaksi[df_transaksi['jenis'] == 'Pengeluaran'].copy()

# Dominasi kategori
dominasi = df_keluar.groupby('kategori')['nominal'].agg(['sum', 'count', 'mean']).reset_index()
dominasi.columns = ['kategori', 'total', 'jumlah_transaksi', 'rata_rata']
dominasi['proporsi (%)'] = (dominasi['total'] / dominasi['total'].sum() * 100).round(1)
dominasi = dominasi.sort_values('total', ascending=False).reset_index(drop=True)
print('Dominasi pengeluaran per kategori:')
print(dominasi.to_string(index=False))

# Deteksi outlier pengeluaran per kategori menggunakan IQR
print('\nDeteksi pengeluaran tidak wajar (IQR 3x per kategori):')
for kat in df_keluar['kategori'].unique():
    sub = df_keluar[df_keluar['kategori'] == kat]['nominal']
    Q1, Q3 = sub.quantile(0.25), sub.quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 3 * IQR
    outlier = sub[sub > upper]
    print(f'  {kat}: {len(outlier)} transaksi melebihi batas Rp{upper:,.0f}')

fig, ax = plt.subplots(figsize=(10, 4.5))
warna_kat = [PALET_LIST[i] for i in range(len(dominasi))]
bars = ax.bar(dominasi['kategori'], dominasi['total'], color=warna_kat, alpha=0.9)
ax.set_title('BQ4: Total Pengeluaran per Kategori', fontsize=13)
ax.set_ylabel('Total Pengeluaran (Rp)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp{x/1_000_000:.0f}M'))
ax.margins(y=0.12)
for i, row in dominasi.iterrows():
    ax.text(i, row['total'] + (dominasi['total'].max()*0.01),
            f"{row['proporsi (%)']}%", ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

### BQ 5: Bagaimana proyeksi arus kas warung 2 hingga 4 minggu ke depan berdasarkan pola historis?

In [ ]:
# EDA pendahuluan untuk forecasting: visualisasi pola time series sebelum masuk ke model

# Agregasi pemasukan harian (semua warung)
ts_harian = df_transaksi[
    df_transaksi['jenis'] == 'Pemasukan'
].groupby('tanggal')['nominal'].sum().reset_index()
ts_harian.columns = ['ds', 'y']
ts_harian = ts_harian.sort_values('ds')

# Rolling average 7 hari untuk visualisasi tren
ts_harian['rolling_7d'] = ts_harian['y'].rolling(7, min_periods=1).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot time series asli
axes[0].plot(ts_harian['ds'], ts_harian['y'], color=PALET['neutral'], linewidth=0.9, alpha=0.7, label='Harian')
axes[0].plot(ts_harian['ds'], ts_harian['rolling_7d'], color=PALET['primary'], linewidth=2, label='Rolling 7 hari')
axes[0].set_title('BQ5: Tren Pemasukan Harian (dasar untuk Cash Flow Forecasting)', fontsize=12)
axes[0].set_ylabel('Total Pemasukan (Rp)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp{x/1_000_000:.1f}M'))
axes[0].legend(frameon=True, loc='upper right')

# Autocorrelation plot sederhana untuk cek seasonality
lags = range(1, 31)
autocorr = [ts_harian['y'].autocorr(lag=l) for l in lags]
axes[1].bar(lags, autocorr, color=PALET['secondary'], alpha=0.9)
axes[1].axhline(y=0, color='#555555', linewidth=0.8)
axes[1].set_title('Autocorrelation Pemasukan Harian (lag 1-30 hari)', fontsize=12)
axes[1].set_xlabel('Lag (hari)')
axes[1].set_ylabel('Korelasi')

plt.tight_layout()
plt.show()

print('Data harian siap untuk digunakan sebagai input model Cash Flow Forecasting.')
print(f'Jumlah titik data: {len(ts_harian)} hari')
print(f'Rentang: {ts_harian["ds"].min().date()} s.d. {ts_harian["ds"].max().date()}')
print('\nHead DataFrame untuk model:')
print(ts_harian[['ds', 'y']].head())

---
## 7. Ringkasan Temuan EDA
Kesimpulan dari seluruh analisis yang akan menjadi dasar tahap pemodelan.

In [ ]:
print('==== RINGKASAN TEMUAN EDA ====')
print()
print('BQ1 - Laba Bersih')
print(f'  Rata-rata laba bersih bulanan: Rp{cashflow["laba_bersih"].mean():,.0f}')
print(f'  Bulan dengan laba tertinggi  : {rata_bulanan.loc[rata_bulanan["laba_bersih"].idxmax(), "bulan"]}')
print()
print('BQ2 - Margin vs Volume')
print(f'  Produk margin tertinggi : {df_bcg_prep.loc[df_bcg_prep["margin_pct"].idxmax(), "nama_produk"]} ({df_bcg_prep["margin_pct"].max():.1f}%)')
print(f'  Produk volume tertinggi : {df_bcg_prep.loc[df_bcg_prep["qty_terjual"].idxmax(), "nama_produk"]} ({df_bcg_prep["qty_terjual"].max():,.0f} unit)')
print()
print('BQ3 - Pola Waktu')
print(f'  Jam penjualan tertinggi : {jam_puncak}:00 - {jam_puncak+1}:00')
print(f'  Hari penjualan tertinggi: {hari_puncak}')
print()
print('BQ4 - Pengeluaran')
print(f'  Kategori pengeluaran terbesar: {dominasi.iloc[0]["kategori"]} ({dominasi.iloc[0]["proporsi (%)"]:.1f}%)')
print()
print('BQ5 - Forecasting')
print(f'  Data time series siap: {len(ts_harian)} titik data harian')
print('  Lanjut ke notebook forecasting untuk membangun model prediksi.')